In [21]:

"""Hana Research: tolvaptan 導入理由候補の抽出（Step 4）

ind_date（サムスカ導入日）の30日前〜14日後のカルテから、
導入理由候補に関する記載をキーワード検索し、候補カルテをCSVに保存する。

注意：このスクリプトは候補カルテを抽出するStep 4のみを実装する。
患者単位の0/1表を作成するStep 5は、添付されたコードに含まれていない。
"""

import re
import sqlite3
import unicodedata
from pathlib import Path

import pandas as pd

DB_PATH = Path("/Users/muna/Hana_research/data/db/Hana_Research.db")
OUTPUT_DIR = Path("/Users/muna/Hana_research/src/tolvaptan_study/")
REASON_CANDIDATE_FILE = OUTPUT_DIR / "step4_reason_candidates.csv"

# 導入日前30日〜導入後14日
DAYS_BEFORE = 30
DAYS_AFTER = 14
NEGATION_WINDOW = 10

# 否定表現。NFKC正規化後は全角括弧などが半角に統一される。
NEGATION_REGEX = re.compile(
    r"(なし|無し|なく|無く|認めず|認めない|否定|ない|\(\s*[-－]\s*\))"
)

# 条件・指示文に見られる表現。キーワード直後の短い範囲だけを判定する。
CONDITION_REGEX = re.compile(
    r"(時|場合|あれば|ようなら|際|連絡|ください|下さい|有無|確認|チェック|リスク|注意|既往|疑い|可能性|恐れ|おそれ)"
)
# 読点・句点・改行をまたがず、最大10文字を許容
GAP = r"[^、。,;；\r\n]{0,10}"
WORSEN = r"(?:出現|増悪|悪化|増強|強まっ|強くな|出た|新規|増加)"

REASON_PATTERNS = {
    "weight_gain": [
        r"体重増加",
        r"体重.{0,10}増加",
        r"体重.{0,10}増",
        r"体重上昇",
        r"WEIGHT GAIN",
        r"\bBW.{0,10}増加",
        r"\bBW\s*(?:UP|↑)",
    ],
    "edema": [
        r"浮腫",
        r"むくみ",
        r"下腿浮腫",
        r"EDEMA",
    ],
    "dyspnea": [
        rf"(?:呼吸苦|呼吸困難|息切れ|息苦し[いさ]?|起座呼吸){GAP}{WORSEN}",
        rf"労作時呼吸困難{GAP}{WORSEN}",
        rf"(?<![A-Z])DOE(?![A-Z]){GAP}{WORSEN}",
        rf"(?:DYSPNEA|ORTHOPNEA){GAP}(?:WORSEN|INCREAS|ONSET|NEW)",
        rf"(?:新たに|新規){GAP}(?:呼吸苦|呼吸困難|息切れ|息苦し|起座呼吸)",
        r"(?:WORSENING|NEW\s+ONSET|NEW)\s+(?:DYSPNEA|ORTHOPNEA)",
    ],
    "fluid_retention": [
        r"水分貯留",
        r"体液貯留",
        r"うっ血",
        r"肺うっ血",
        r"CONGESTION",
    ],
    "diuretic_resistance": [
        r"利尿(?:剤|薬)?(?:抵抗|反応不良|効果不十分|効果不良|不良|無効)",
        r"REFRACTORY",
        r"ラシックス.{0,4}増量",
        r"ラシックス.{0,4}漸増",
        r"フロセミド.{0,4}増量",
        r"フロセミド.{0,4}漸増",
        r"ループ利尿薬.{0,4}増量",
        r"ループ利尿剤.{0,4}増量",
        r"利尿薬.{0,2}IV化",
        r"利尿剤.{0,2}IV化",
        r"利尿薬.{0,2}静注化",
        r"利尿剤.{0,2}静注化",
    ],
    "hyponatremia": [
        r"低Na血症",
        r"低ナトリウム血症",
        r"HYPONATREMIA",
    ],
    "pleural_effusion": [
        r"胸水",
        r"PLEURAL EFFUSION",
    ],
    "ascites": [
        r"腹水",
        r"ASCITES",
    ],
}

def has_unnegated_match(text: str, combined_regex: re.Pattern, negation_window: int = NEGATION_WINDOW) -> bool:
    """一致箇所の直後に否定・条件表現がない場合にTrueを返す。"""
    for match in combined_regex.finditer(text):
        # 検索語そのものに否定語が含まれる場合も除外
        if NEGATION_REGEX.search(match.group(0)):
            continue

        tail = text[match.end(): match.end() + negation_window]
        # 句読点・改行より後は、同じ記載の直後とはみなさない
        tail = re.split(r"[、。,;；\r\n]", tail)[0]

        if NEGATION_REGEX.search(tail):
            continue
        if CONDITION_REGEX.search(tail):
            continue
        return True

    return False


def run_step4() -> None:
    print("#" * 70)
    print(f"STEP 4: TOLVAPTAN REASON CANDIDATES (-{DAYS_BEFORE}d / +{DAYS_AFTER}d)")
    print("#" * 70)

    if not DB_PATH.is_file():
        raise FileNotFoundError(f"データベースが見つかりません: {DB_PATH}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(DB_PATH) as conn:
        tolvaptan = pd.read_sql_query(
            """
            SELECT Patient_ID, Study_ID, ind_date
            FROM tolvaptan_study
            WHERE ind_date IS NOT NULL
            """,
            conn,
        )

        if tolvaptan.empty:
            raise ValueError("tolvaptan_study に ind_date が登録された対象患者がいません。")

        tolvaptan["Patient_ID"] = tolvaptan["Patient_ID"].astype(str)
        tolvaptan["ind_date"] = pd.to_datetime(tolvaptan["ind_date"], errors="coerce")
        tolvaptan = tolvaptan.dropna(subset=["ind_date"])

        patient_ids = tolvaptan["Patient_ID"].dropna().unique().tolist()
        if not patient_ids:
            raise ValueError("有効な Patient_ID がありません。")

        placeholders = ",".join(["?"] * len(patient_ids))
        karte = pd.read_sql_query(
            f"""
            SELECT Patient_ID, Study_ID, visit_datetime, karte_text
            FROM karte
            WHERE Patient_ID IN ({placeholders})
              AND karte_text IS NOT NULL
              AND karte_text != ''
            """,
            conn,
            params=patient_ids,
        )

    karte["Patient_ID"] = karte["Patient_ID"].astype(str)
    karte["visit_date"] = pd.to_datetime(
        karte["visit_datetime"].astype(str).str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})", expand=False),
        errors="coerce",
    )

    # 患者IDで導入日を付与
    tolvaptan_merge = tolvaptan[["Patient_ID", "Study_ID", "ind_date"]].rename(
        columns={"Study_ID": "Study_ID_tolva"}
    )
    karte = karte.merge(tolvaptan_merge, on="Patient_ID", how="inner")
    karte["days_from_ind"] = (karte["visit_date"] - karte["ind_date"]).dt.days

    karte_window = karte[
        karte["visit_date"].notna()
        & karte["ind_date"].notna()
        & (karte["days_from_ind"] >= -DAYS_BEFORE)
        & (karte["days_from_ind"] <= DAYS_AFTER)
    ].copy()

    print(f"対象患者数: {len(tolvaptan):,}")
    print(f"取得カルテ件数: {len(karte):,}")
    print(f"導入日前{DAYS_BEFORE}日〜導入後{DAYS_AFTER}日のカルテ件数: {len(karte_window):,}")

    karte_window["karte_text_search"] = karte_window["karte_text"].astype(str).map(
        lambda s: unicodedata.normalize("NFKC", s)
    ).str.upper()

    hits = []
    for reason, patterns in REASON_PATTERNS.items():
        regex = re.compile("|".join(patterns), flags=re.IGNORECASE)
        mask = karte_window["karte_text_search"].apply(
            lambda text: has_unnegated_match(text, regex)
        )
        tmp = karte_window.loc[
            mask,
            [
                      "Study_ID_tolva",
                "Patient_ID",
                "ind_date",
                "visit_date",
                "days_from_ind",
                "visit_datetime",
                "karte_text",
            ],
        ].copy()
        tmp["reason"] = reason
        hits.append(tmp)

   # 該当ゼロの項目があっても、出力列を維持して保存できるようにする
    if hits:
        result = pd.concat(hits, ignore_index=True)
    else:
        result = pd.DataFrame()

    if result.empty:
        result = pd.DataFrame(
            columns=[
                "Study_ID_tolva", "Patient_ID", "ind_date", "visit_date",
                "days_from_ind", "visit_datetime", "karte_text", "reason"
            ]
        )

    result = result.rename(columns={"Study_ID_tolva": "Study_ID"})
    result = result[
        [
            "Study_ID", "Patient_ID", "ind_date", "reason", "visit_date",
            "days_from_ind", "visit_datetime", "karte_text"
        ]
    ]
    result.to_csv(REASON_CANDIDATE_FILE, index=False, encoding="utf-8-sig")

    print(f"候補カルテCSV: {REASON_CANDIDATE_FILE}")
    print("理由別ヒット件数:")
    if result.empty:
        print("該当なし")
    else:
        print(result["reason"].value_counts().reindex(REASON_PATTERNS.keys(), fill_value=0))


if __name__ == "__main__":
    run_step4()

######################################################################
STEP 4: TOLVAPTAN REASON CANDIDATES (-30d / +14d)
######################################################################
対象患者数: 214
取得カルテ件数: 26,951
導入日前30日〜導入後14日のカルテ件数: 3,279
候補カルテCSV: /Users/muna/Hana_research/src/tolvaptan_study/step4_reason_candidates.csv
理由別ヒット件数:
reason
weight_gain             429
edema                  1521
dyspnea                 181
fluid_retention         230
diuretic_resistance      35
hyponatremia             73
pleural_effusion        515
ascites                 202
Name: count, dtype: int64


In [22]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

df_check = pd.read_csv(REASON_CANDIDATE_FILE, dtype={"Study_ID": str, "Patient_ID": str})

# (a) edemaとしてヒットした記載の前後文脈
e = df_check[df_check["reason"] == "edema"].copy()
e["context"] = e["karte_text"].str.extract(r"(.{0,20}(?:浮腫|むくみ).{0,20})", expand=False)
e[["Study_ID", "days_from_ind", "context"]].head(30)

,Study_ID,days_from_ind,context
429,P656628,0,下腿、臀部浮腫：重度\r
430,P656628,-7,両足の蜂窩織炎。むくみもひどく仙骨部、腸骨部に発赤もあり。\r
431,P208041,4,下腿浮腫：軽度\r
432,P208041,3,下腿浮腫についても改善している。\r
433,P208041,1,循環：両下肢・背腹部に浮腫(左下肢中等度、右下肢軽度)、３日診察時
434,P208041,-4,下腿浮腫：中等度\r
435,P208041,-11,下腿浮腫：中等度\r
436,P208041,-13,循環：両下肢・背腹部に浮腫ある。（左下肢中等度、右下肢軽度）足が冷
437,P208041,-17,下腿浮腫：中等度\r
438,P208041,-20,循環：両下肢・背腹部に浮腫+１。下肢は足趾まで浮腫ある。\r


In [23]:
# (b) dyspneaとしてヒットした記載の前後文脈
d = df_check[df_check["reason"] == "dyspnea"].copy()
d["context"] = d["karte_text"].str.extract(r"(.{0,20}(?:呼吸苦|呼吸困難|息切れ|息苦し|起座呼吸|DOE).{0,20})", expand=False)
d[["Study_ID", "days_from_ind", "context"]].head(30)

,Study_ID,days_from_ind,context
1950,P013689,13,体動時の息切れは改善したように見える、と。\r
1951,P318755,-18,体位変換時呼吸困難感軽度出現あり。\r
1952,P013689,-1,息苦しい症状は本人の話だとない。\r
1953,P013689,-24,安静時は息切れない。\r
1954,P432407,13,息切れも改善。痰は出せている。\r
1955,P432407,4,息切れも改善。痰も減っている。\r
1956,P446266,0,ご家族へMTし、今後呼吸苦など強くなれば緩和主体のケアへ切り替えを
1957,P012835,13,：酸素５?で９1％前後（安静）安静時には呼吸苦訴え聞かれないが、ト労作時には呼吸苦出現
1958,P012835,9,：酸素５?で９４％前後（安静）安静時には呼吸苦訴え聞かれないが、トイレ食事の労作時には
1959,P012835,6,：酸素５?で８８％前後（安静）安静時には呼吸苦訴え聞かれないが、トイレ食事の労作時には


In [24]:
# (c) weight_gainとしてヒットした記載の前後文脈
w = df_check[df_check["reason"] == "weight_gain"].copy()
w["context"] = w["karte_text"].str.extract(r"(.{0,20}(?:体重|BW).{0,20})", expand=False)
w[["Study_ID", "days_from_ind", "context"]].head(30)

,Study_ID,days_from_ind,context
0,P013689,6,娘様：とくには変わらないですが、体重は増えていませんよ\r
1,P013689,-4,体重増加\r
2,P130887,7,体重は33.6㎏。(12/24 34.4 と
3,P130887,5,体重23日より2kg増 訪問時体重再測定34.4と1kg増 下腿浮腫なし。
4,P432407,-15,体重増加、下腿浮腫\r
5,P725289,4,体重：47.5(9/6)→51kg(9/21
6,P725289,0,下肢浮腫増悪 体重増加\r
7,P725289,-9,下肢浮腫増悪 体重増加\r
8,P451462,14,体重：59.5㎏(＋3.5㎏/週)。\r
9,P451462,7,体重：56.2kg ← 60kg(6/6)、


In [ ]:
# ============================================================
# チェック & 確認プロンプト
# ============================================================

print("=" * 70)
print("データベース反映前の確認")
print("=" * 70)
print("\n先頭10例:")
print(patient_info[["Study_ID", "Patient_ID"] + reason_list].head(10).to_string(index=False))
print(f"\n理由別の該当人数:")
for reason in reason_list:
    print(f"  {reason}: {patient_info[reason].sum():3d} 人")

confirm = input("\nこの内容でデータベースに反映しますか？ (yes/no): ")

if confirm.lower() == "yes":
    # DB書き込み処理
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    for idx, row in patient_info.iterrows():
        study_id = row["Study_ID"]
        update_sql = f"""
        UPDATE tolvaptan_study
        SET {', '.join([f'{col} = ?' for col in reason_list])}
        WHERE Study_ID = ?
        """
        params = [row[col] for col in reason_list] + [study_id]
        cursor.execute(update_sql, params)
    conn.commit()
    print(f"✓ {len(patient_info)}件のレコードを更新しました")
    conn.close()
else:
    print("CSVのみ出力して、DBへの反映はスキップしました")

In [25]:
import unicodedata

GAP = r"[^、。,;；\n]{0,10}"        # 読点・改行をまたがない。小数点「.」は許可（2.5kg対策）
WORSEN = r"(?:出現|増悪|悪化|増強|強まっ|強くな|出た|新規|増加)"

NEGATION_REGEX = re.compile(r"(なし|無し|なく|無く|認めず|認めない|否定|ない|\(\s*-\s*\))")
CONDITION_REGEX = re.compile(r"(時|場合|あれば|ようなら)")   # 「増悪時は〜」等の指示文

REASON_PATTERNS = {
    "weight_gain": [
        rf"体重{GAP}(?:増加|増|上昇)",
        r"WEIGHT GAIN",
        rf"(?<![A-Z])BW{GAP}(?:増加|増|UP|↑)",
    ],
    "edema": [
        rf"(?:浮腫|むくみ){GAP}{WORSEN}",
        rf"(?:新た|新規)に?{GAP}(?:浮腫|むくみ)",
        r"(?:WORSENING|NEW)\s+EDEMA",
        r"EDEMA.{0,10}(?:WORSEN|PROGRESS|INCREAS)",
    ],
    "dyspnea": [
        rf"(?:呼吸苦|呼吸困難|息切れ|息苦し[いさ]?|起座呼吸){GAP}{WORSEN}",
        rf"(?<![A-Z])DOE(?![A-Z]){GAP}{WORSEN}",
        rf"(?:新たに|新規){GAP}(?:呼吸苦|呼吸困難|息切れ|息苦し|起座呼吸)",
        r"(?:WORSENING|NEW\s+ONSET|NEW)\s+(?:DYSPNEA|ORTHOPNEA)",
    ],
    # fluid_retention 以降は現状のまま（#9, #10 の方針決定後に修正）
}

def _has_unnegated_match(text, combined_regex, negation_window=NEGATION_WINDOW):
    for m in combined_regex.finditer(text):
        tail = re.split(r"[、。\n]", text[m.end(): m.end() + negation_window])[0]
        if (NEGATION_REGEX.search(m.group(0))      # キーワードと増悪語の間の否定
                or NEGATION_REGEX.search(tail)      # 直後の否定（句読点で打ち切り）
                or CONDITION_REGEX.search(tail)):   # 条件・指示文
            continue
        return True
    return False

In [26]:
import re
import unicodedata

NEGATION_WINDOW = 10

GAP = r"[^、。,;；\r\n]{0,10}"   # 読点・改行をまたがない（小数点「.」は許可）
WORSEN = (r"(?:出現|増悪|悪化|増強|増加|↑|強まっ|強くな|ひどく|増え|進行"
          r"|拡大|広が|目立ってき|出てき|出た|新規)")
EDEMA = r"(?:(?<!リンパ)(?<!リンパ性)(?<!薬剤性)(?<!本態性)(?<!全周性)浮腫|むくみ)"
DYSP = r"(?:呼吸苦|呼吸困難|息切れ|息苦し[いさ]?|起座呼吸)"

NEGATION_REGEX = re.compile(r"(なし|無し|なく|無く|認めず|認めない|否定|ない|\(\s*-\s*\))")
CONDITION_REGEX = re.compile(
    r"(時|場合|あれば|ようなら|際|連絡|ください|下さい|有無|確認|チェック"
    r"|リスク|注意|既往|疑い|可能性|恐れ|おそれ)")

REASON_PATTERNS = {
    "weight_gain": [
        rf"体重{GAP}(?:増加|増|上昇)",
        r"WEIGHT GAIN",
        rf"(?<![A-Z])BW{GAP}(?:増加|増|UP|↑)",
    ],
    "edema": [
        rf"{EDEMA}{GAP}{WORSEN}",
        rf"(?:新た|新規)に?{GAP}{EDEMA}",
        r"(?:WORSENING|NEW)\s+EDEMA",
        r"EDEMA.{0,10}(?:WORSEN|PROGRESS|INCREAS)",
    ],
    "dyspnea": [
        rf"{DYSP}{GAP}{WORSEN}",
        rf"(?<![A-Z])DOE(?![A-Z]){GAP}{WORSEN}",
        rf"(?:新たに|新規){GAP}{DYSP}",
        r"(?:WORSENING|NEW\s+ONSET|NEW)\s+(?:DYSPNEA|ORTHOPNEA)",
    ],
    # ---- 以下5理由は元コードのまま ----
    "fluid_retention": [
        r"水分貯留",
        r"体液貯留",
        r"うっ血",
        r"肺うっ血",
        r"CONGESTION",
    ],
    "diuretic_resistance": [
        r"利尿(?:剤|薬)?(?:抵抗|反応不良|効果不十分|効果不良|不良|無効)",
        r"REFRACTORY",
        r"ラシックス.{0,4}増量",
        r"ラシックス.{0,4}漸増",
        r"フロセミド.{0,4}増量",
        r"フロセミド.{0,4}漸増",
        r"ループ利尿薬.{0,4}増量",
        r"ループ利尿剤.{0,4}増量",
        r"利尿薬.{0,2}IV化",
        r"利尿剤.{0,2}IV化",
        r"利尿薬.{0,2}静注化",
        r"利尿剤.{0,2}静注化",
    ],
    "hyponatremia": [
        r"低Na血症",
        r"低ナトリウム血症",
        r"HYPONATREMIA",
    ],
    "pleural_effusion": [
        r"胸水",
        r"PLEURAL EFFUSION",
    ],
    "ascites": [
        r"腹水",
        r"ASCITES",
    ],
}


def _has_unnegated_match(text, combined_regex, negation_window=NEGATION_WINDOW):
    for m in combined_regex.finditer(text):
        tail = re.split(r"[、。\r\n]", text[m.end(): m.end() + negation_window])[0]
        if (NEGATION_REGEX.search(m.group(0))      # キーワードと増悪語の間の否定
                or NEGATION_REGEX.search(tail)      # 直後の否定（句読点で打ち切り）
                or CONDITION_REGEX.search(tail)):   # 条件・指示文
            continue
        return True
    return False